In [0]:
%run ../utils/00_silver_utils

In [0]:
# Widget params; run_id comes from bronze via get_shared_run_id so this notebook's audit rows share bronze's run_id.
dbutils.widgets.text("silver_config_path", "/Volumes/cdr_silver/config/silver_config/silver_config.json")
dbutils.widgets.text("table_filter", "")

silver_config_path = dbutils.widgets.get("silver_config_path")
table_filter_raw = dbutils.widgets.get("table_filter").strip()
table_filter = {t.strip().lower() for t in table_filter_raw.split(",") if t.strip()} if table_filter_raw else None

dbutils.widgets.text("run_id", "")
run_id_param = dbutils.widgets.get("run_id").strip()
run_id = run_id_param if run_id_param else str(uuid.uuid4())
if not run_id_param:
    logger.warning(f"No run_id widget value supplied -- generating standalone run_id={run_id}")

logger.info(f"run_id              = {run_id}")
logger.info(f"silver_config_path  = {silver_config_path}")
logger.info(f"table_filter        = {table_filter or 'ALL TABLES FOR THIS NOTEBOOK'}")

In [0]:
# Loads silver_config.json and filters it down to the tables tagged silver_notebook == "core_reference".
with open(silver_config_path, "r") as f:
    SILVER_CONFIG = json.load(f)

LAYER = "core_reference"
tables_for_this_notebook = [
    t for t, cfg in SILVER_CONFIG.items() if cfg["silver_notebook"] == LAYER
]
if table_filter:
    tables_for_this_notebook = [t for t in tables_for_this_notebook if t in table_filter]

logger.info(f"Processing {len(tables_for_this_notebook)} table(s) for layer '{LAYER}': {tables_for_this_notebook}")

ensure_silver_audit_table(spark)

In [0]:
# Bespoke transform for admissions: fixes ~175 rows where admittime is after dischtime, silently (no audit flag column).
def clean_admissions(df):
    # ~175 rows have admittime > dischtime, which should never happen. Fix (no flag column -- confirmed no flag needed):
    #   1) if edregtime and edouttime both exist and edregtime < edouttime: admittime = edregtime, dischtime = edouttime
    #   2) otherwise: keep admittime as-is, set dischtime = admittime + 1 hour
    bad_row = col("admittime") > col("dischtime")
    ed_valid = col("edregtime").isNotNull() & col("edouttime").isNotNull() & (col("edregtime") < col("edouttime"))

    df = df.withColumn(
        "dischtime",
        when(bad_row & ed_valid, col("edouttime"))
        .when(bad_row & ~ed_valid, col("admittime") + expr("INTERVAL 1 HOURS"))
        .otherwise(col("dischtime"))
    )
    df = df.withColumn(
        "admittime",
        when(bad_row & ed_valid, col("edregtime")).otherwise(col("admittime"))
    )
    return df

In [0]:
# Bespoke transform for drgcodes: makes description single-valued per (drg_type, drg_code) in place -- no separate dimension table.
from pyspark.sql import Window
from pyspark.sql.functions import first as _first

def clean_drgcodes(df):
    # Each (drg_type, drg_code) must map to a single description -- no separate dim table (confirmed).
    # Tiebreak rule (not specified beyond "prefer non-null" -- chosen here for determinism across reruns):
    # alphabetically-first non-null description per (drg_type, drg_code) group.
    w = Window.partitionBy("drg_type", "drg_code").orderBy(col("description").asc_nulls_last())
    return df.withColumn("description", _first(col("description"), ignorenulls=True).over(w))

In [0]:
# Maps each table needing bespoke logic (not expressible as a generic cleaning_rules entry) to its function.
SPECIAL_TRANSFORMS = {
    "admissions": clean_admissions,
    "drgcodes": clean_drgcodes
}

In [0]:
# Runs every table for this layer: read bronze, clean, write/merge to silver, collect an audit row, then write the audit log.
audit_rows = []
success_count = 0
failure_count = 0

for table_name in tables_for_this_notebook:
    audit_row = process_table(
        spark, table_name, SILVER_CONFIG, run_id, layer=LAYER,
        special_transform=SPECIAL_TRANSFORMS.get(table_name),
    )
    audit_rows.append(audit_row)
    if audit_row.status == "SUCCESS":
        success_count += 1
    else:
        failure_count += 1

write_audit_rows(spark, audit_rows)

logger.info(
    f"Layer '{LAYER}' complete | run_id={run_id} | {success_count} succeeded | "
    f"{failure_count} failed | {len(tables_for_this_notebook)} total tables"
)
if failure_count > 0:
    logger.warning(f"{failure_count} table(s) failed -- check {SILVER_AUDIT_TABLE} (run_id = {run_id})")